# Kizzasi Audio Processing Pipeline

This notebook demonstrates Kizzasi's audio-oriented capabilities:

1. Simulate a multi-tone audio signal
2. Apply μ-law (MuLaw) 8-bit encoding / decoding
3. Extract MFCC-style spectral features
4. Streaming tokenization with a sliding window
5. Anomaly detection via prediction error + constraint guardrails

All `kizzasi` calls are wrapped in `try/except ImportError` so the notebook runs stand-alone even when the compiled wheel is not installed.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import spectrogram as scipy_spectrogram
from scipy.fft import dct

try:
    import kizzasi
    KIZZASI_AVAILABLE = True
    print(f'kizzasi {kizzasi.__version__} loaded')
except ImportError:
    KIZZASI_AVAILABLE = False
    print('kizzasi not installed — numpy-stub mode active')

SAMPLE_RATE = 16000  # Hz
DURATION    = 1.0    # seconds
N_SAMPLES   = int(SAMPLE_RATE * DURATION)

rng = np.random.default_rng(42)

## 1. Simulate Multi-Tone Audio

We combine three sinusoids (220 Hz, 440 Hz, 880 Hz) with slight noise — a simplified chord that exercises the tokenizer across multiple spectral bands.

In [ ]:
t = np.linspace(0, DURATION, N_SAMPLES, endpoint=False)

signal = (
    0.5  * np.sin(2 * np.pi * 220  * t) +
    0.35 * np.sin(2 * np.pi * 440  * t) +
    0.15 * np.sin(2 * np.pi * 880  * t) +
    0.03 * rng.standard_normal(N_SAMPLES)
).astype(np.float32)

# Normalise to [-1, 1]
signal /= np.abs(signal).max() + 1e-8

print(f'Signal: {N_SAMPLES} samples @ {SAMPLE_RATE} Hz')
print(f'Range : [{signal.min():.4f}, {signal.max():.4f}]')

fig, ax = plt.subplots(figsize=(12, 2.5))
ax.plot(t[:400], signal[:400], linewidth=0.8, color='steelblue')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('Multi-tone audio signal (first 400 samples)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/kizzasi_audio_signal.png', dpi=120)
plt.show()

## 2. μ-Law Encoding / Decoding

μ-law (ITU-T G.711) compresses audio into 8 bits by applying a logarithmic transfer function that emphasises low-amplitude detail — the standard codec for telephony and WaveNet-style models.

Kizzasi's `MuLawCodec` is exposed directly as `kizzasi.MuLawCodec` — a PyO3 binding over `kizzasi_tokenizer::MuLawCodec`, the same codec `kizzasi-tokenizer`'s WASM bindings expose to JavaScript. Used below when the compiled wheel is installed, with a pure-numpy fallback otherwise so this notebook still runs standalone.

In [ ]:
MU = 255.0  # 8-bit mu-law parameter

def mulaw_encode(x: np.ndarray, mu: float = MU) -> np.ndarray:
    """Compress float32 signal in [-1, 1] to uint8 mu-law tokens."""
    x = np.clip(x, -1.0, 1.0)
    encoded = np.sign(x) * np.log1p(mu * np.abs(x)) / np.log1p(mu)
    return ((encoded + 1.0) / 2.0 * mu + 0.5).astype(np.uint8)

def mulaw_decode(tokens: np.ndarray, mu: float = MU) -> np.ndarray:
    """Expand uint8 mu-law tokens back to float32 in [-1, 1]."""
    x = tokens.astype(np.float32) / mu * 2.0 - 1.0
    return np.sign(x) * (1.0 / mu) * ((1.0 + mu) ** np.abs(x) - 1.0)

# --- Use kizzasi.MuLawCodec (a real PyO3 binding) if the wheel is
# installed, else fall back to the numpy implementation above ---
if KIZZASI_AVAILABLE and hasattr(kizzasi, 'MuLawCodec'):
    codec = kizzasi.MuLawCodec(bits=8)
    tokens = codec.quantize_array(signal).astype(np.uint8)
    reconstructed = codec.dequantize_array(tokens.astype(np.int32))
else:
    tokens = mulaw_encode(signal)
    reconstructed = mulaw_decode(tokens)

snr_db = 10 * np.log10(
    np.mean(signal ** 2) / (np.mean((signal - reconstructed) ** 2) + 1e-12)
)
print(f'Tokens    : dtype={tokens.dtype}, range=[{tokens.min()}, {tokens.max()}]')
print(f'Codec SNR : {snr_db:.1f} dB')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=False)

seg = slice(0, 300)
axes[0].plot(t[seg], signal[seg], color='steelblue', linewidth=1)
axes[0].set_title('Original float32')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

axes[1].step(np.arange(300), tokens[seg], color='darkorange', linewidth=0.8, where='mid')
axes[1].set_title('μ-law uint8 tokens')
axes[1].set_ylabel('Token value')
axes[1].grid(True, alpha=0.3)

axes[2].plot(t[seg], reconstructed[seg], color='mediumseagreen', linewidth=1)
axes[2].plot(t[seg], signal[seg], color='steelblue', linewidth=0.6, alpha=0.4, linestyle='--')
axes[2].set_title(f'Reconstructed (SNR = {snr_db:.1f} dB)')
axes[2].set_ylabel('Amplitude')
axes[2].set_xlabel('Time (s) / Sample index')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/kizzasi_mulaw.png', dpi=120)
plt.show()

## 3. MFCC-Style Feature Extraction

Mel-Frequency Cepstral Coefficients (MFCCs) compress spectral information into a compact feature vector per frame. Kizzasi's `kizzasi-io` crate includes MFCC extraction in Rust; here we replicate the pipeline in numpy/scipy for illustration.

In [ ]:
N_MFCC    = 13
N_MELS    = 40
FRAME_LEN = 512    # ~32 ms at 16 kHz
HOP_LEN   = 160    # ~10 ms at 16 kHz

def hz_to_mel(hz: float) -> float:
    return 2595.0 * np.log10(1.0 + hz / 700.0)

def mel_to_hz(mel: float) -> float:
    return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)

def mel_filterbank(n_fft: int, n_mels: int, sr: int) -> np.ndarray:
    """Return (n_mels, n_fft//2+1) triangular mel filterbank."""
    mel_min = hz_to_mel(0.0)
    mel_max = hz_to_mel(sr / 2.0)
    mel_points = np.linspace(mel_min, mel_max, n_mels + 2)
    hz_points = np.array([mel_to_hz(m) for m in mel_points])
    bins = np.floor((n_fft + 1) * hz_points / sr).astype(int)
    filterbank = np.zeros((n_mels, n_fft // 2 + 1))
    for m in range(1, n_mels + 1):
        for k in range(bins[m - 1], bins[m]):
            filterbank[m - 1, k] = (k - bins[m - 1]) / (bins[m] - bins[m - 1] + 1e-8)
        for k in range(bins[m], bins[m + 1] + 1):
            filterbank[m - 1, k] = (bins[m + 1] - k) / (bins[m + 1] - bins[m] + 1e-8)
    return filterbank

def extract_mfcc(audio: np.ndarray, sr: int, n_fft: int,
                 hop: int, n_mels: int, n_mfcc: int) -> np.ndarray:
    """Return (n_mfcc, n_frames) MFCC matrix."""
    window = np.hanning(n_fft)
    frames = []
    for start in range(0, len(audio) - n_fft + 1, hop):
        frame = audio[start:start + n_fft] * window
        spectrum = np.abs(np.fft.rfft(frame)) ** 2
        frames.append(spectrum)
    power_spec = np.stack(frames, axis=1)           # (n_fft//2+1, n_frames)
    fb = mel_filterbank(n_fft, n_mels, sr)
    mel_energy = np.dot(fb, power_spec)             # (n_mels, n_frames)
    log_mel = np.log(mel_energy + 1e-8)
    mfccs = dct(log_mel, type=2, axis=0, norm='ortho')[:n_mfcc]
    return mfccs

mfccs = extract_mfcc(signal, SAMPLE_RATE, FRAME_LEN, HOP_LEN, N_MELS, N_MFCC)
print(f'MFCC matrix shape: {mfccs.shape}  (n_mfcc x n_frames)')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(
    mfccs, aspect='auto', origin='lower',
    extent=[0, DURATION, 0, N_MFCC],
    cmap='inferno'
)
plt.colorbar(im, ax=ax, label='MFCC coefficient')
ax.set_xlabel('Time (s)')
ax.set_ylabel('MFCC index')
ax.set_title(f'{N_MFCC}-coefficient MFCC — multi-tone audio signal')
plt.tight_layout()
plt.savefig('/tmp/kizzasi_mfcc.png', dpi=120)
plt.show()

## 4. Streaming Tokenization (Sliding Window)

Kizzasi is designed for real-time streaming. The `Predictor.step()` API processes one sample at a time with O(1) cost. Here we simulate a streaming loop that accumulates tokens and computes a running prediction error.

In [ ]:
STREAM_WARMUP = 500
STREAM_LEN    = 1000

if KIZZASI_AVAILABLE:
    stream_cfg = kizzasi.Config.audio(sample_rate=SAMPLE_RATE)
    stream_pred = kizzasi.Predictor(stream_cfg)

    predictions = []
    errors = []

    # Warm-up phase: feed signal without recording predictions
    for i in range(STREAM_WARMUP):
        x = np.array([signal[i]], dtype=np.float32)
        stream_pred.step(x)

    # Streaming phase
    for i in range(STREAM_WARMUP, STREAM_WARMUP + STREAM_LEN):
        x = np.array([signal[i]], dtype=np.float32)
        pred = stream_pred.step(x)      # returns numpy array shape (1,)
        true_next = signal[i + 1] if i + 1 < len(signal) else 0.0
        predictions.append(float(pred[0]))
        errors.append(abs(true_next - float(pred[0])))

    predictions = np.array(predictions)
    errors      = np.array(errors)
else:
    # Numpy stub: add small Gaussian noise to simulate prediction error
    seg = signal[STREAM_WARMUP : STREAM_WARMUP + STREAM_LEN]
    predictions = seg + rng.normal(0, 0.02, size=STREAM_LEN).astype(np.float32)
    errors      = np.abs(signal[STREAM_WARMUP + 1 : STREAM_WARMUP + STREAM_LEN + 1] - predictions)

print(f'Streaming MAE: {errors.mean():.5f}')

## 5. Constraint Guardrails

Kizzasi supports neuro-symbolic guardrails that clip or reject predictions violating physical constraints. For audio this could enforce a safe amplitude envelope; for sensors it could enforce physically plausible ranges.

In [ ]:
if KIZZASI_AVAILABLE:
    # Attach an amplitude guardrail: clip predictions to [-0.95, 0.95]
    amplitude_guard = kizzasi.ConstraintSpec(
        name='amplitude',
        min_val=-0.95,
        max_val= 0.95,
        hard_reject=False  # soft: clip rather than raise
    )
    stream_pred.set_guardrails([amplitude_guard])
    print(f'Guardrails active: {stream_pred.has_guardrails()}')

    # After attaching guardrails, subsequent step() calls enforce them
    x = np.array([0.99], dtype=np.float32)   # near-clipping input
    guarded_out = stream_pred.step(x)
    print(f'Guarded prediction for input=0.99: {guarded_out[0]:.4f} (expected <= 0.95)')

    stream_pred.clear_guardrails()
    print(f'Guardrails active after clear: {stream_pred.has_guardrails()}')
else:
    print('(stub) ConstraintSpec amplitude guardrail [-0.95, 0.95] demonstrated')
    print('(stub) set_guardrails() / clear_guardrails() / has_guardrails()')

In [ ]:
# Visualise streaming prediction quality
steps = np.arange(STREAM_LEN)
true_seg = signal[STREAM_WARMUP : STREAM_WARMUP + STREAM_LEN]

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(steps, true_seg,    label='Ground truth', color='steelblue', linewidth=1)
axes[0].plot(steps, predictions, label='Prediction',   color='coral',     linewidth=1, alpha=0.8)
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Streaming Prediction — Multi-tone Audio')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(steps, errors, color='mediumseagreen', linewidth=0.8)
axes[1].axhline(errors.mean(), color='firebrick', linewidth=1, linestyle='--',
                label=f'MAE = {errors.mean():.5f}')
axes[1].set_ylabel('|Error|')
axes[1].set_xlabel('Streaming step')
axes[1].set_title('Absolute Prediction Error')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/kizzasi_streaming.png', dpi=120)
plt.show()

## Summary

This notebook demonstrated:

- **μ-law tokenization**: lossily compress audio to 8-bit integers (SNR typically 30–40 dB for speech)
- **MFCC features**: 13 cepstral coefficients per 10 ms frame, matching the input of classical ASR systems
- **Streaming inference**: O(1) per-step cost using SSM hidden state — no growing attention window
- **Guardrails**: attach `ConstraintSpec` objects to enforce amplitude envelopes at inference time

Continue to `anomaly_detection.ipynb` for a full train-then-detect workflow.